# 4. Regimes, in real time

**Question this notebook answers**
Do the macro regimes survive being estimated without hindsight, and do they line up with recessions
anyone would recognise? Answer is a fold-by-fold agreement rate between real-time and full-sample
labels, plus a comparison against the three NBER recessions in the sample.

**What v1 did**
`unsupervised.ipynb:501` runs PCA on all 309 rows, `:998` builds the 4-D embedding from all 309,
`:1392` predicts labels for all 309. `:1455` smooths with `.rolling(window=3, center=True)`. Cluster
count 10 was chosen by silhouette over `range(2, 11)`, and `:1961` maps the ten names to cluster
indices in alphabetical order.

**Why that was wrong**
Four compounding problems. The clustering saw the test period before labelling it, and those labels
were then fed to Phase 2 as features. A centred three-month window reads month t+1, so the smoothed
label at any month depends on the future. The chosen k sat on the edge of its own search grid and
contradicted the notebook's own inertia elbow, producing a five-observation cluster for a component
with roughly fifteen parameters. And the names were assigned alphabetically rather than from the
centroids, which is why "Crisis Peak" has one of the lowest mean spreads in the sample.

**What this notebook does instead**
Refits PCA and the clustering inside each walk-forward fold on training data only, uses k in the
4 to 5 range indicated by the elbow, filters rather than centre-smooths, derives names from
centroids ordered by a stress index, and checks the labels against NBER recession dates.

**Predicted effect on results — recorded before running**
Real-time labels should disagree with full-sample labels most around turning points, which is
exactly where a regime label would be most useful — I expect agreement well below 100%, plausibly
in the 70 to 85% range. Regime features should add little or nothing to forecast accuracy once the
spread's own history is present. Fewer, larger clusters should map onto NBER recessions more
recognisably than ten did.

**Inputs** → **Outputs**
`data/v2/panel_pit.parquet` → in-memory fold-wise regime labels consumed by notebook 05; figures to
`results/figures/`.

---
## 4.1 Choose k honestly

Compute silhouette and inertia across a k range that extends well past the candidate, on training data only. Look at whether the silhouette peak is a peak or the edge of the grid, and where the elbow actually falls.

---
## 4.2 Fit PCA and clustering inside one fold

Fit the scaler, PCA and clustering on a single fold's training block, then transform and predict the test block. Nothing is fitted on test data.

---
## 4.3 Repeat across all folds

Loop `src.splits.walk_forward_folds`, refitting the whole regime pipeline per fold. Record labels with the fold that produced them.

---
## 4.4 Real-time versus hindsight labels

Compare the fold-wise labels against labels from a single full-sample fit. Look at where they disagree and whether the disagreements cluster around turning points.

---
## 4.5 Filtered versus smoothed

Compare a causal (trailing) smoother against v1's centred window. Quantify how many months the centred version relabels using information from the following month.

---
## 4.6 Name regimes from their centroids

Rank centroids by a stress index built from unemployment, growth and the spread, and assign names in that order. Print the mapping alongside the centroid values so the names are checkable.

---
## 4.7 Validate against NBER recession dates

Cross-tabulate labels against `src.config_v2.NBER_RECESSIONS`. Look for the stress-ranked regimes concentrating in the 2001, 2007-09 and 2020 windows.

---
## 4.8 Regime persistence and cluster sizes

Report how long regimes last and how large the smallest cluster is. Look for clusters large enough to estimate, unlike v1's five-observation component.